# Apply Post Effects

Apply reflections, filtering, normalization, tail trimming, and fade-out to `data/processed/mix.wav`, then save the final 16-bit PCM result as `outputs/audio.wav`.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import butter, filtfilt

from voice_to_birdcall.paths import OUTPUT_DIR, PROCESSED_DATA_DIR

SR = 44_100

mix_path = PROCESSED_DATA_DIR / "mix.wav"
output_path = OUTPUT_DIR / "audio.wav"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load Event Mix

Read the floating-point intermediate without changing its amplitude before applying the post-processing chain.

In [2]:
def finite_audio(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if not np.all(np.isfinite(y)):
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def normalize_audio(y: np.ndarray) -> np.ndarray:
    y = finite_audio(y)
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    return y if peak < 1e-8 else y / peak


if not mix_path.exists():
    raise FileNotFoundError(f"Event mix not found: {mix_path}")

mix, sample_rate = sf.read(str(mix_path), dtype="float32", always_2d=False)
if sample_rate != SR:
    raise ValueError(f"Unexpected sample rate for {mix_path.name}: {sample_rate}")
if mix.ndim > 1:
    mix = np.mean(mix, axis=1)
mix = finite_audio(mix)

print(f"Loaded event mix: {len(mix) / SR:.2f} seconds")

Loaded event mix: 6.22 seconds


# Apply Echo and Filtering

Add a 90 ms reflection followed by 180, 340, and 520 ms echoes, then apply a zero-phase 5.2 kHz low-pass filter.

In [3]:
def lowpass_filter(y: np.ndarray, cutoff: float) -> np.ndarray:
    if len(y) < 32:
        return y
    b, a = butter(4, cutoff / (SR / 2), btype="lowpass")
    padlen = min(3 * max(len(a), len(b)), len(y) - 1)
    return filtfilt(b, a, y, padlen=padlen)


def dreamy_echo(y: np.ndarray) -> np.ndarray:
    result = y.copy()
    for delay, gain in zip([0.18, 0.34, 0.52], [0.28, 0.18, 0.10]):
        samples = int(delay * SR)
        if samples < len(y):
            result[samples:] += y[:-samples] * gain
    return normalize_audio(lowpass_filter(result, cutoff=5_200))


output = mix.copy()
delay_90ms = int(0.09 * SR)
if delay_90ms < len(output):
    output[delay_90ms:] += output[:-delay_90ms] * 0.25

output = normalize_audio(dreamy_echo(output)) * 0.85

# Trim and Save Final Audio

Trim the tail below -50 dB with 80 ms of padding, apply a 60 ms fade-out, and write `outputs/audio.wav`.

In [4]:
peak = float(np.max(np.abs(output))) if len(output) else 0.0
if peak > 1e-8:
    silence_threshold = peak * (10 ** (-50 / 20))
    audible_samples = np.flatnonzero(np.abs(output) > silence_threshold)
    if len(audible_samples):
        tail_padding = int(0.08 * SR)
        final_sample = min(
            len(output),
            int(audible_samples[-1]) + 1 + tail_padding,
        )
        output = output[:final_sample]

fade_samples = min(int(0.06 * SR), len(output) // 2)
if fade_samples > 0:
    output[-fade_samples:] *= np.linspace(
        1.0,
        0.0,
        fade_samples,
        dtype=np.float32,
    )

sf.write(str(output_path), finite_audio(output), SR, subtype="PCM_16")

print(f"Saved final audio to: {output_path}")
print(f"Output duration: {len(output) / SR:.2f} seconds")
display(Audio(output, rate=SR))

Saved final audio to: C:\Users\morph\GitHub\voice-to-birdcall\outputs\audio.wav
Output duration: 4.57 seconds
